# 06 - Model Evaluation

### Formula 1 Race Prediction & Driver Analytics

---

| | |
|---|---|
| **Input**    | `data/processed/f1_features_2018_2026.csv`  |
|              | `outputs/models/` (saved models)            |
| **Output**   | `outputs/06_*.png` (evaluation charts)      |
| **Previous** | Notebook 05 — Model Training                |
| **Next**     | Notebook 07 — Driver Cards                  |

---

## What This Notebook Does

In Notebook 05 we selected the best model for each target based on the
**validation set (2024)**. The test set (2025–2026) was never touched.

This notebook performs **final evaluation on the held-out test set.**

We answer:
- How well does the position model predict race finishing order?
- How well does the classifier predict who scores points?
- How well does the points model predict championship points earned?
- Where does the model fail — which drivers, circuits, conditions?
- What is the custom Prediction Reward Score?

---

## Why Final Evaluation on a Held-Out Test Set Matters

During Notebook 05 we made many decisions based on validation performance:
- Which model family to use
- Which hyperparameters to set
- Which features to include

Each decision was implicitly "tuned" to the validation set (2024).
This means validation metrics are optimistic — they reflect our choices.

The test set (2025–2026) has never influenced any decision.
Its metrics are the most honest estimate of real-world performance.


## 📐 Evaluation Framework

### Metrics by Target

**Target: `position` (Regression)**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MAE | mean(|predicted - actual|) | Average error in positions |
| RMSE | sqrt(mean((pred - actual)²)) | Penalises large errors more |
| R² | 1 - SS_res/SS_tot | % of variance explained (1.0 = perfect) |

**Target: `points_finish` (Classification)**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| Accuracy | correct / total | Overall correctness |
| Precision | TP / (TP + FP) | Of predicted "scores points" — how many did? |
| Recall | TP / (TP + FN) | Of actual "scores points" — how many caught? |
| F1 | 2 · P · R / (P + R) | Balance of precision and recall |

**Target: `points` (Regression)**
Same metrics as position — MAE, RMSE, R².

---

### Custom Prediction Reward Score

A domain-specific scoring system that reflects how useful the predictions
are for a fantasy F1 or betting context:

| Prediction Accuracy | Points Awarded |
|--------------------|----------------|
| Exact position | +10 |
| Within ±1 position | +7 |
| Within ±2 positions | +5 |
| Within ±3 positions | +3 |
| Correct points finish (yes/no) | +2 |
| Correct podium prediction | +3 |

This metric rewards predictions that are close even when not exact —
which standard MAE already captures, but this makes it more interpretable
to a non-technical F1 audience.


# Import Libs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

F1_RED    = '#E8002D'

F1_DARK = '#15151E',
F1_SILVER = '#C0C0C0',

os.makedirs('../output', exist_ok=True)
print('Imports comlete')

# Load Dataset

In [ ]:
df = pd.read_csv('../data/processed/f1_features_2018_2026.csv')

print('Dataset loaded')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Seasons: {df.season.min()} - {df.season.max()}')

## Load Metadata

In [ ]:

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# ── Best models ───────────────────────────────────────────
best_pos_model = load_pkl('../output/models/best_position_model.pkl')
best_clf_model = load_pkl('../output/models/best_clf_model.pkl')
best_pts_model = load_pkl('../output/models/best_points_model.pkl')

# ── Scaler and features ───────────────────────────────────
scaler     = load_pkl('../output/models/feature_scaler.pkl')
FEATURES   = load_pkl('../output/models/feature_list.pkl')
metadata   = load_pkl('../output/models/model_metadata.pkl')

import pprint
pprint.pprint(metadata)
print('Models loaded:')
print(f'  Position model:      {metadata["position_model"]["name"]}')
print(f'  Points finish model: {metadata["points_finish_model"]["name"]}')
print(f'  Points model:        {metadata["points_model"]["name"]}')
print()

def get_any(d, *keys, default='N/A'):
    """Пробует несколько вариантов названия ключа."""
    for k in keys:
        if k in d:
            return d[k]
    return default

print('Validation metrics (from Notebook 05):')

pos_mae = get_any(metadata['position_model'], 'val_mae', 'MAE')
print(f'  Position  - Val MAE: {pos_mae}')

clf_f1 = get_any(metadata['points_finish_model'], 'val_f1', 'val_F1', 'F1', 'f1', 'f1_score')
print(f'  Pts Finish - Val F1: {clf_f1}')

pts_mae = get_any(metadata['points_model'], 'val_mae', 'MAE')
print(f'  Points    - Val MAE: {pts_mae}')


split_info = load_pkl('../output/models/train_test_split.pkl')

print('Models loaded:')
print(f'  Position model:      {metadata["position_model"]["name"]}')
print(f'  Points finish model: {metadata["points_finish_model"]["name"]}')
print(f'  Points model:        {metadata["points_model"]["name"]}')
print()
print('Validation metrics (from Notebook 05):')
print(f'  Position  - Val MAE: {metadata["position_model"]["val_mae"]:.4f}')
print(f'  Pts Finish - Val F1: {metadata["points_finish_model"]["val_f1"]:.4f}')
print(f'  Points    - Val MAE: {metadata["points_model"]["val_mae"]:.4f}')

# Reconstruct Test Set

In [ ]:
TEST_SEASONS = split_info['test_seasons']
VAL_SEASONS = split_info['val_seasons']
TRAIN_SEASONS = split_info['train_seasons']

test_mask = df.season.isin(TEST_SEASONS)
val_mask = df.season.isin(VAL_SEASONS)
train_mask = df.season.isin(TRAIN_SEASONS)

X_test = df[test_mask][FEATURES].values
X_val = df[val_mask][FEATURES].values
X_train = df[train_mask][FEATURES].values

#Scaled ver for linear models
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

#target vectors
y_test = {
    'position': df[test_mask]['position'].values,
    'points_finish': df[test_mask]['points_finish'].values,
    'points': df[test_mask]['points'].values
}

#keep test df for error analysis(circuit, driver breakdowns)
df_test = df[test_mask].copy().reset_index(drop=True)

print('Test set reconstructed:')
print(f'Seasons: {TEST_SEASONS}')
print(f'Rows: {X_test.shape[0]:,}')
print(f'Features: {X_test.shape[1]}')

print('\nSeason breakdown of test set:')
print(df_test.groupby('season').size().to_string())



# Generate All Predictions

In [ ]:
def is_linear(model):
    name = type(model).__name__
    return name in ('LinearRegression', 'LogisticRegression')

# Position Predictions
if is_linear(best_pos_model):
    pos_preds = best_pos_model.predict(X_test_scaled)
else:
    pos_preds = best_pos_model.predict(X_test)
pos_preds = np.clip(pos_preds, 1, 20)

#points finish predictions
if is_linear(best_clf_model):
    clf_preds = best_clf_model.predict(X_test_scaled)
    clf_proba = best_clf_model.predict_proba(X_test_scaled)[:, 1]
else:
    clf_preds = best_clf_model.predict(X_test)
    clf_proba = best_clf_model.predict_proba(X_test)[:, 1]

#points predictions
if is_linear(best_clf_model):
    pts_preds = best_pts_model.predict(X_test_scaled)
else:
    pts_preds = best_pts_model.predict(X_test)
pts_preds = np.clip(pts_preds, 0, 26)

#attach pred to test df
df_test['pred_position'] = pos_preds
df_test['pred_points_finish'] = clf_preds
df_test['pred_points_finish'] = pts_preds
df_test['pred_points'] = pts_preds
df_test['pred_proba'] = clf_proba
df_test['pos_error'] = pos_preds - df_test['position']
df_test['pts_error'] = pts_preds - df_test['points']

print('Predictions generated for test set')
print(f'Position predictions: min={pos_preds.min():.2f} max={pos_preds.max():.2f}')
print(f'Points predictions: min={pts_preds.min():.2f} max={pts_preds.max():.2f}')
print(f'Points finish preds: {(clf_preds==1).sum()} predicted to score'
      f'({(clf_preds==1).mean()*100:.1f}%)')

# Helper Functions

In [ ]:
def regression_report(y_true, y_pred, target_name, val_mae=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f'MAE: {mae:.4f}', end='')
    if val_mae:
        diff = mae - val_mae
        flag = '✅' if diff <= 0.2 else '⚠️'
        print(f' (val: {val_mae:.4f}, diff: {diff:+.4f}) {flag}', end = '')

    
    print(f'\nRMSE: {rmse:.4f}')
    print(f'R2: {r2:.4f}')
    return {
        'target': target_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    }

In [ ]:
def classification_report_custom(y_true, y_pred, val_f1=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec= recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f'Accuracy: {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall {rec:.4f}')
    print(f'F1 Score {f1:.4f}', end='')
    if val_f1:
        diff = f1 - val_f1
        flag = '✅' if abs(diff) <=0.05 else '⚠️'
    print()
    return {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    }


test_results = {}

print('Helper functions defined')

# Section 1 - Position Model Evaluation


### What Good Performance Looks Like

| MAE | Interpretation |
|-----|---------------|
| < 2.5 | Excellent — within 2–3 positions on average |
| 2.5 – 3.5 | Good — useful for fantasy F1 and race prediction |
| 3.5 – 5.0 | Moderate — better than naive but limited utility |
| > 5.0 | Poor — barely better than always predicting P10 |

---

### Why Position Prediction Is Hard

F1 finishing positions are determined by:
- Car pace (captured by constructor features ✅)
- Driver skill (captured by driver form features ✅)
- Strategy calls (NOT in our data ❌)
- Safety car timing (NOT in our data ❌)
- First-lap incidents (NOT in our data ❌)
- Weather changes (NOT in our data ❌)

A model with MAE ≈ 3.0 is performing near the theoretical limit
of what structured historical data can achieve without live race data.

In [ ]:
print(f'Position Model - {metadata["position_model"]["name"]}')
print(f'Test Set: seasons {TEST_SEASONS}')
print("="*65 +"\n")

pos_metrics = regression_report(
    y_test['position'],
    pos_preds,
    target_name='position',
    val_mae=metadata['position_model']['val_mae']
)

test_results['position'] = pos_metrics

print('\nPrediction distribution:')
print(f'Actual positions - mean: {y_test["position"].mean():.2f}'
      f'std: {y_test["position"].std():2f}')
print(f'Predicted positions - mean: {pos_preds.mean():.2f}'
      f'std: {pos_preds.std():.2f}')


#error breakdown by zone
errors = np.abs(pos_preds - y_test['position'])
print('Error breakdown:')
print(f'Exact (error = 0): {(errors == 0).sum():>5}'
      f'({(errors == 0).mean()*100:.1f}%)')
print(f'Within +-1 position: {(errors <=1).sum():>5}'
      f'({(errors <=1).mean()*100:.1f}%)')
print(f'Within +-2 positions: {(errors<=2).sum():>5}'
      f'({(errors<=2).mean()*100:.1f}%)')
print(f'Within +-3 positions: {(errors<=3).sum():>5}'
      f'({(errors<=3).mean()*100:.1f}%)')
print(f'Error > 5 positions: {(errors > 5).sum():>5}'
      f'({(errors  >5).mean()*100:.1f}%)')

## Position - Actual vs Predicted Scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))

ax1 = axes[0]
ax1.scatter(
    y_test['position'], pos_preds,
    alpha=0.15, s=10, color='#3498DB', edgecolors='none'
)
ax1.plot([1, 20], [1, 20], color=F1_RED, linewidth=2,
         linestyle='--', label='Perfect prediction')

#highest top3 pred
top3_mask = y_test['position']<=3
ax1.scatter(
    y_test['position'][top3_mask],
    pos_preds[top3_mask],
    alpha=0.5, s=20, color='gold',
    edgecolors='black', linewidth=0.3,
    label='Actual P1-P3', zorder = 5
)

ax1.set_xlabel('Actual Finishing Position', fontsize=12)
ax1.set_ylabel('Predicted Finishing Position', fontsize=12)
ax1.set_title('Actual vs Predicted Position\n(Test Set 2026-2026)',
              fontsize=13, fontweight='bold')
ax1.set_xlim(0.5, 20.5)
ax1.set_ylim(0.5, 20.5)
ax1.legend(fontsize=9)
ax1.invert_xaxis()
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#right - error distribution histogram
ax2 = axes[1]
errors_signed = pos_preds - y_test['position']
ax2.hist(errors_signed, bins=35, color='#3468DB',
         edgecolor='white', linewidth=0.5, alpha=0.85)
ax2.axvline(0, color=F1_RED, linewidth=2,
            linestyle='--', label='Zero error')
ax2.axvline(errors_signed.mean(), color='green', linewidth =1.5,
            label=f'Mean error: {errors_signed.mean():.2f}')
ax2.axvline(np.median(errors_signed), color='green', linewidth=1.5,
            linestyle=':', label=f'Median error: {np.median(errors_signed):.2f}')

ax2.set_xlabel('Prediction Error (Predicted - Actual positions)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Distribution of Position Prediction Errors', 
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle(f'Position Model Evaluation - {metadata["position_model"]["name"]}'
             f'- MAE: {pos_metrics["MAE"]:.3f}',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('../output/06_position_actual_vs_predicted.png', 
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Position - Error by Finishing Zone

In [ ]:
zones = {
    'P1-P3 (Podium)': (1, 3),
    'P4-P5 (Near miss)': (4, 5),
    'P6-P10 (Points)': (6, 10),
    'P11-P15 (Midfield)': (11, 15),
    'P16-P20 (Tail)': (16, 20),
}

zone_names, zone_maes, zone_counts = [], [], []

for zone_label, (lo, hi) in zones.items():
    mask = (y_test['position'] >=lo) & (y_test['position'] <= hi) 
    if mask.sum() > 0:
        mae_zone = mean_absolute_error(
            y_test['position'][mask], pos_preds[mask]
        )
        zone_names.append(zone_label)
        zone_maes.append(mae_zone)
        zone_counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(11,5))

colors_zone=['gold', '#F5A623', '#27AE60', '#3498DB', '#9B59B6']
bars = ax.bar(zone_names, zone_maes, color=colors_zone,
              edgecolor='white', linewidth=0.6, width=0.65)

for i, (bar, mae, cnt) in enumerate(zip(bars, zone_maes, zone_counts)):
    ax.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() + 0.03,
        f'{mae:.2f}\n(n={cnt})',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.axhline(pos_metrics['MAE'], color='black', linestyle='--',
           linewidth=1.5, label=f'Overall MAE: {pos_metrics["MAE"]:.3f}')
ax.set_ylabel('MAE (positions)', fontsize=12)
ax.set_title('Position Prediction MAE by Finishing Zone\n'
             '(Where is the model most/least accurate?)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, max(zone_maes) +0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_position_mae_by_zone.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
print('KEY INSIGHT')
worst_zone=zone_names[np.argmax(zone_maes)]
best_zone=zone_names[np.argmin(zone_maes)]
print(f'Hardest zone to predict: {worst_zone} (MAE={max(zone_maes):.2f})')
print(f'Easiest zone to predict: {best_zone} (MAE={min(zone_maes):.2f})')

## Position - Error by Driver

In [ ]:
driver_errors = (
    df_test.groupby('driver_name')
    .apply(lambda g: mean_absolute_error(g['position'], g['pred_position']))
    .reset_index()
    .rename(columns={0: 'mae'})
    .sort_values('mae', ascending=False)
    .reset_index(drop=True)
)


#Only include drivers with 5+ test races
driver_counts = df_test.groupby('driver_name').size().reset_index(name='count')
driver_errors = driver_errors.merge(driver_counts, on='driver_name')
driver_errors = driver_errors[driver_errors['count'] >= 5].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13,8))

bar_colors= [
    F1_RED if m > pos_metrics['MAE'] * 1.3 else
    '#F5A623' if m > pos_metrics['MAE'] else
    '#27AE60'
    for m in driver_errors['mae']
]


ax.barh(
    driver_errors['driver_name'],
    driver_errors['mae'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.4,
    height=0.75
)

for i, (mae, cnt) in enumerate(
    zip(driver_errors['mae'], driver_errors['count'])
):
    ax.text(
        mae + 0.03, i,
        f'{mae:.2f} (n={cnt})',
        va='center', ha='left', fontsize=9
    )

ax.axvline(pos_metrics['MAE'], color='black', linestyle='--',
           linewidth=1.5, label=f'Overall MAE: {pos_metrics["MAE"]:.3f}')
ax.set_xlabel('MAE (positions)', fontsize=12)
ax.set_title('Position Prediction MAE by Driver - Test Set\n'
             '(Higher = harder to predict)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, driver_errors['mae'].max() +1.5)
ax.legend(fontsize=9)

ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_position_mae_driver.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
print('KEY INSIGHT')
hardest=driver_errors.iloc[0]
easiest=driver_errors.iloc[-1]
print(f'Hardest to predict: {hardest['driver_name']} (MAE={hardest["mae"]:.2f})')
print(f'Easiest to predict: {easiest['driver_name']} (MAE={easiest['mae']:.2f})')

## Position - Error by Circuit

In [ ]:
circuit_errors = (
    df_test.groupby('circuit_id')
    .apply(lambda g: mean_absolute_error(g['position'], g['pred_position']))
    .reset_index()
    .rename(columns={0: 'mae'})
    .sort_values('mae', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(13,8))

bar_colors_c = [
    F1_RED if m > pos_metrics['MAE'] *1.4 else
    '#F5A623' if m > pos_metrics['MAE'] else
    '#27AE60'
    for m in circuit_errors['mae']
]

ax.barh(
    circuit_errors['circuit_id'],
    circuit_errors['mae'],
    color=bar_colors_c,
    edgecolor='white',
    linewidth=0.4,
    height=0.75
)

for i, mae in enumerate(circuit_errors['mae']):
    ax.text(
        mae +0.03, i,
        f'{mae:.2f}',
        va='center', ha ='left', fontsize=9
    )

ax.axvline(pos_metrics['MAE'], color='black', linestyle='--',
           linewidth=1.5, label=f'Overall MAE: {pos_metrics["MAE"]:.3f}')

ax.set_xlabel('MAE (positions)', fontsize=12)
ax.set_title('Position Prediction MAE by Circuit - Test Set\n'
             '(High MAE = chaotic circuit, low MAE = predictable circuit)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, circuit_errors['mae'].max() +1.2)
ax.legend(fontsize=9)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_position_mae_by_circuit.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
print('KEY INSIGHT')
print(f'Most unpredictable: {circuit_errors.iloc[0]['circuit_id']}'
      f'(MAE={circuit_errors.iloc[0]["mae"]:.2f})')
print(f'Most predictable: {circuit_errors.iloc[-1]['circuit_id']}'
      f'(MAE={circuit_errors.iloc[-1]["mae"]:.2f})')
print('\nUnpredictable circuits typically have:')
print('high DNF rates, safety cars, or strong weather variability')


# Section 2 - Points Finish Classification Evaluation

### Why This Target Matters

Correctly predicting who scores points is the most actionable prediction:
- Fantasy F1 scoring is based on points finishes
- Team strategy decisions revolve around the P10 threshold
- Sponsor value is largely determined by consistent points scoring

**Class balance in test set:**
Approximately 50% of race starts result in points (top 10 of 20 drivers).
This means a naive model that always predicts "scores points" achieves
~50% accuracy — our F1 score must meaningfully beat this.

---

### Precision vs Recall Trade-off

**High Precision:** When we predict a driver scores points, we are
usually right. But we may miss some actual points scorers.

**High Recall:** We catch most actual points scorers, but we may
over-predict — incorrectly labelling some non-scorers as scorers.

**F1 Score** balances both. For most use cases (fantasy F1, betting),
a balanced F1 is the right target.


## Points Finish - Core Metrics

In [ ]:
print(f'POINTS FINISH MODEL — {metadata["points_finish_model"]["name"]}')
print(f'Test Set: seasons {TEST_SEASONS}')

clf_metrics = classification_report_custom(
    y_test['points_finish'],
    clf_preds,
    val_f1=metadata['points_finish_model']['val_f1']
)
test_results['points_finish'] = clf_metrics

print('\nDetailed classification report:')
print(classification_report(
    y_test['points_finish'],
    clf_preds,                     # ← убран дубликат
    target_names=['No Points (P11-P20)', 'Scored Points (P1-P10)']
))

print('Actual class distribution in test set:')
vals, counts = np.unique(y_test['points_finish'], return_counts=True)
total = len(y_test['points_finish'])
for v, c in zip(vals, counts):
    label = 'Scored Points' if v == 1 else 'No Points'
    print(f'  {label}: {c:>5} ({c/total*100:.1f}%)')

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test['points_finish'], clf_preds)
cm_pct = cm / cm.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#left - raw counts
ax1 = axes[0]
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted\nNo Points', 'Predicted\nPoints'],
    yticklabels=['Actual\nNo Points', 'Actual\nPoints'],
    ax=ax1, cbar=False,
    annot_kws={'size': 14, 'weight': 'bold'}
)

ax1.set_title('Confusion Matrix - Raw Counts', fontsize=13, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=11)
ax1.set_xlabel('Predicted', fontsize=11)

#add tn/fp/fn/tp labels
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax1.text(j +0.5, i+0.75, labels[i][j],
                 ha='center', fontsize=10, color='grey')
        
#tight - percentage
ax2 = axes[1]
sns.heatmap(
    cm_pct *100, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=['Predicted\nNo Points', 'Predicted\nPoints'],
    yticklabels=['Actual\nNo Points', 'Actual\nPoints'],
    ax=ax2, cbar=False,
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax2.set_title('Confusion Matrix - (%) of All Predictions', fontsize=13, fontweight='bold')
ax2.set_ylabel('Actual', fontsize=11)
ax2.set_xlabel('Predicted', fontsize=11)

tn, fp, fn, tp = cm.ravel()
fig.suptitle(
    f'Points Finish Classification - {metadata['points_finish_model']['name']}\n'
    f'F1={clf_metrics['F1']:.3f}'
    f'Precision={clf_metrics['Precision']:.3f}'
    f'Recall={clf_metrics['Recall']:.3f}',
    fontsize=13, fontweight='bold', y=1.02
)

fig.tight_layout()
fig.savefig('../output/06_confusion_matrix.png', 
            dpi=150, bbox_inches='tight')

plt.show()
plt.close(fig)



## Roc Curve

In [ ]:
fpr, tpr, threholds = roc_curve(y_test['points_finish'], clf_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8,7))

ax.plot(fpr, tpr, color=F1_RED, linewidth=2.5,
        label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0,1], [0,1], color='grey', linewidth=1.5,
        linestyle='--', label='Random classifier (AUC = 0.50)')

#mark current threhold operating point
current_thresh_idx = np.argmin(
    np.abs(threholds - 0.5)

) if len(threholds) > 0 else 0
ax.scatter(
    fpr[current_thresh_idx], tpr[current_thresh_idx],
    s=120, color='black', zorder=5, label='Threshold = 0.50'
)

ax.fill_between(fpr, tpr, alpha=0.08, color=F1_RED)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title(f'ROC Curve - Points Finish Classifier\n'
             f'AUC = {roc_auc:.3f}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)


print(f'AUC Score: {roc_auc:.4f}')
if roc_auc > 0.85:
    print('✅ Excellent classifier (AUC > 0.85)')
elif roc_auc > 0.75:
    print('✅ Good classifier (AUC > 0.75)')
elif roc_auc > 0.65:
    print('⚠️ Moderate classifier (AUC > 0.65)')
else:
    print('❌ Weak classifier (AUC < 0.65)')
    

# Section 3 - Championship Points Model Evaluation

### Why This Target Is the Hardest

The F1 points system creates a uniquely difficult regression problem:

```
P1  = 25 pts    P6  = 8 pts    P11 = 0 pts
P2  = 18 pts    P7  = 6 pts    P12 = 0 pts
P3  = 15 pts    P8  = 4 pts    ...
P4  = 12 pts    P9  = 2 pts    P20 = 0 pts
P5  = 10 pts    P10 = 1 pt
```

Notice two things:
1. **Non-linear jumps:** P1→P2 = 7 pts gap, P9→P10 = 1 pt gap
2. **Hard cliff:** P10→P11 = 1 pt → 0 pts (massive relative drop)

A model predicting P11 when the actual result is P10 makes a positional
error of 1 — but a points error of 1 vs 0. The model must learn
that the P10/P11 boundary is disproportionately important.

---

### Interpreting MAE in Points

| MAE (pts) | Interpretation |
|-----------|---------------|
| < 2.0 | Excellent — very accurate points prediction |
| 2.0 – 4.0 | Good — useful for championship simulation |
| 4.0 – 6.0 | Moderate — captures general scoring patterns |
| > 6.0 | Poor — not much better than predicting mean |


In [ ]:
print(f'Points Model - {metadata['points_model']['name']}')
print(f'Test Set: seasons {TEST_SEASONS}')

In [ ]:
pts_metrics = regression_report(
    y_test['points'],
    pts_preds,
    target_name='points',
    val_mae=metadata['points_model']['val_mae']
)
test_results['points'] = pts_metrics

print('Prediction distribution:')
print(f'Actual points - mean: {y_test['points'].mean():.2f}'
      f'std: {y_test['points'].std():.2f}')
print(f'Predicted points - mean: {pts_preds.mean():.2f}'
      f'std: {pts_preds.std():.2f}')

#error breakdown by actual points value
print('\nAccuracy by actual points bracket:')
brackets = [
    ('Race winner (25 pts)', y_test['points'] == 25),
    ('Podium (15-18 pts)', (y_test['points'] >= 15) & (y_test['points'] < 25)),
    ('Points (1-12 pts)', (y_test['points'] >= 1) & (y_test['points'] < 15)),
    ('No points (0 pts)', y_test['points'] == 0),
]

for label, mask in brackets:
    if mask.sum() > 0:
        mae_b = mean_absolute_error(
            y_test['points'][mask], pts_preds[mask]
        )
        print(f'{label:<30} n={mask.sum():>4} MAE={mae_b:.2f}')

## Points - Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── LEFT: Scatter ──────────────────────────────────────────
ax1 = axes[0]

scatter_colors = [
    'gold'     if p == 25 else
    '#E67E22'  if p >= 15 else
    '#27AE60'  if p >= 1  else
    '#95A5A6'
    for p in y_test['points']
]

ax1.scatter(
    y_test['points'], pts_preds,
    c=scatter_colors, alpha=0.3, s=12, edgecolors='none'
)
max_val = max(y_test['points'].max(), pts_preds.max())
ax1.plot([0, max_val], [0, max_val],
         color=F1_RED, linewidth=2, linestyle='--',
         label='Perfect prediction')

ax1.set_xlabel('Actual Points', fontsize=12)
ax1.set_ylabel('Predicted Points', fontsize=12)
ax1.set_title('Actual vs Predicted Championship Points\n(Test Set)',
              fontsize=13, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_el = [
    Patch(facecolor='gold',    label='Race winner (25)'),
    Patch(facecolor='#E67E22', label='Podium (15–18)'),
    Patch(facecolor='#27AE60', label='Points (1–12)'),
    Patch(facecolor='#95A5A6', label='No Points (0)'),
]
ax1.legend(handles=legend_el, fontsize=8, loc='upper left')

# ── RIGHT: Avg predicted vs avg actual ─────────────────────
ax2 = axes[1]

# y_test['points'] — numpy array, нет .unique()
# Используем np.unique() вместо .unique()
unique_pts = sorted(np.unique(y_test['points']))

avg_pred_by_pts = [
    pts_preds[y_test['points'] == p].mean()
    for p in unique_pts
]

ax2.plot(unique_pts, unique_pts,
         color='grey', linestyle='--', linewidth=1.5,
         label='Perfect prediction')
ax2.plot(unique_pts, avg_pred_by_pts,
         color=F1_RED, linewidth=2.5, marker='o',
         markersize=5,                              # ← было marksize
         label='Avg predicted')
ax2.fill_between(unique_pts, unique_pts, avg_pred_by_pts,
                 alpha=0.1, color=F1_RED)

ax2.set_xlabel('Actual Points Value', fontsize=12)
ax2.set_ylabel('Average Predicted Points', fontsize=12)
ax2.set_title('Average Predicted vs Actual Points\n'
              '(By actual points bracket)',
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle(
    f'Points Model Evaluation — {metadata["points_model"]["name"]} '  # ← двойные кавычки
    f'— MAE: {pts_metrics["MAE"]:.3f} pts',
    fontsize=14, fontweight='bold', y=1.02
)

fig.tight_layout()
fig.savefig('../output/06_points_actual_vs_predicted.png',  # ← outputs с s
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Points - Error by Driver

In [ ]:
driver_pts_errors = (
    df_test.groupby('driver_name')
    .apply(lambda g: mean_absolute_error(g['points'], g['pred_points']))
    .reset_index()
    .rename(columns={0: 'mae'})
    .merge(
        df_test.groupby('driver_name').size().reset_index(name='count'),
        on='driver_name'
    )
)

driver_pts_errors = (
    driver_pts_errors[driver_pts_errors['count'] >= 5]
    .sort_values('mae', ascending=False)
    .reset_index(drop=True)
)

#also compute avg actual vs predicted points per driver
driver_avg = (
    df_test.groupby('driver_name')
    .agg(
        avg_actual_pts = ('points', 'mean'),
        avg_pred_pts = ('pred_points', 'mean')
    )
    .reset_index()
)
driver_pts_errors = driver_pts_errors.merge(driver_avg, on='driver_name')
driver_pts_errors['bias'] = (
    driver_pts_errors['avg_pred_pts'] - driver_pts_errors['avg_actual_pts']

)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

#left - mae by driver
ax1 = axes[0]
colors_d = [
    F1_RED if m > pts_metrics['MAE'] * 1.4 else
    '#F5A623' if m > pts_metrics['MAE'] else
    '#27AE60'
    for m in driver_pts_errors['mae']
]

ax1.barh(
    driver_pts_errors['driver_name'],
    driver_pts_errors['mae'],
    color=colors_d, edgecolor='white', linewidth=0.4, height=0.75
)

for i, mae in enumerate(driver_pts_errors['mae']):
    ax1.text(mae + 0.05, i, f'{mae:.2f}',
             va='center', ha='left', fontsize=9)
    

ax1.axvline(pts_metrics['MAE'], color ='black', linestyle='--',
            linewidth=1.5, label=f'Overall MAE: {pts_metrics['MAE']:.2f}')
ax1.set_xlabel('MAE (points)', fontsize=11)
ax1.set_title('Points MAE by Driver', fontsize=12, fontweight='bold')
ax1.invert_yaxis()
ax1.legend(fontsize=8)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)


#right - prediction bias by driver

ax2=axes[1]
bias_sorted = driver_pts_errors.sort_values('bias')
colors_bias = [
    F1_RED if b > 0 else '#3498DB'
    for b in bias_sorted['bias']
]

ax2.barh(
    bias_sorted['driver_name'],
    bias_sorted['bias'],
    color=colors_bias, edgecolor='white', linewidth=0.4, height=0.75
)
for i, b in enumerate(bias_sorted['bias']):
    ax2.text(
        b + (0.05 if b >= 0 else -0.05), i,
        f'{b:+2f}',
        va='center',
        ha='left' if b >= 0 else 'right',
        fontsize=9
    )


ax2.axvline(0, color='black', linewidth=1.5)
ax2.set_xlabel('Bias: Predicted - Actual (pts)', fontsize=11)
ax2.set_title('Prediction Bias by Driver\n'
              '(Red = overestimated, Blue = underestimated)',
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Points Model - Driver-Level Error Analysis',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig('../output/06_points_error_by_driver.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
print('KEY INSIGHT')
most_over= driver_pts_errors.loc[driver_pts_errors.bias.idxmax()]
most_under=driver_pts_errors.loc[driver_pts_errors.bias.idxmin()]
print(f'Most overestimated: {most_over['driver_name']}'
      f'(avg +{most_over['bias']:.2f} pts)')
print(f'Most underestimated: {most_under['driver_name']}'
      f'(avg {most_under['bias']:.2f} pts)')

# Section 4 - Custom Prediction Reward Score

### Why Standard Metrics Are Not Enough

MAE = 3.2 positions. What does that mean in F1 terms?
Is predicting P4 when the actual result is P1 acceptable?
Is predicting P11 when the actual is P10 a big mistake?

Standard metrics treat all errors equally. In F1, they are not:

- P1 vs P4: missed a race win — major prediction failure
- P10 vs P11: missed the points boundary — critical error
- P6 vs P8: minor difference within the points zone

---

### Reward Scoring System

Each prediction for each driver in each race earns points:

| Condition | Reward |
|-----------|--------|
| Exact position predicted | +10 |
| Within ±1 position | +7 |
| Within ±2 positions | +5 |
| Within ±3 positions | +3 |
| Correct points_finish prediction | +2 |
| Correct podium prediction (P1–P3) | +3 |
| **Maximum per driver per race** | **15** |

The **Total Reward Score** and **Reward per Race** are our
headline evaluation metrics alongside MAE and F1.


## Compute Reward Score

In [ ]:
def compute_reward(actual_pos, pred_pos, actual_pts_finish,
                   pred_pts_finish, actual_poduim, pred_pos_for_podium):
    reward = 0
    error = abs(round(pred_pos) - actual_pos)

    #position accuracy reward
    if error == 0:
        reward +=10
    elif error <= 1:
        reward +=7
    elif error <=2:
        reward +=5
    elif error <=3:
        reward +=3


    #points finish bonus
    pred_pts_finish_binary = 1 if round(pred_pos) <= 10 else 0
    if pred_pts_finish_binary == actual_pts_finish:
        reward +=2

    #podium prediction bonus
    pred_podium = 1 if round(pred_pos) <= 3 else 0
    actual_poduim_flag = 1 if actual_pos <= 3 else 0
    if pred_podium == actual_poduim_flag:
        reward+=3

    return reward

    

In [ ]:
#Apply to test set
df_test['reward'] = df_test.apply(
    lambda row: compute_reward(
        actual_pos= int(row['position']),
        pred_pos=row['pred_position'],
        actual_pts_finish=int(row['points_finish']),
        pred_pts_finish=int(row['pred_points_finish']),
        actual_poduim=int(row['podium']),
        pred_pos_for_podium=row['pred_position']
    ),
    axis=1

)

#summary statistics
max_possible_per_race = 15
total_reward = df_test['reward'].sum()
max_reward = len(df_test) * max_possible_per_race
reward_pct = total_reward / max_reward *100
avg_reward = df_test['reward'].mean()

## Custom Prediction Reward Score

In [ ]:
print(f'Total reward points: {total_reward:,}')
print(f'Maximum possible: {max_reward:,}')
print(f'Score efficiency: {reward_pct:.1f}%')
print(f'Average reward per race entry: {avg_reward:.2f} / {max_possible_per_race}')

#reward distribution
for pts in [10, 7, 5, 3, 0]:
    if pts > 0:
        label = {10: 'Exact position (+10)',
                 7: 'Within ±1 (+7)',
                 5: 'Within ±2 (+5)',
                 3: 'Within ±3 (+3)'}[pts]
    else:
        label = 'Off by 4+ positions (0)'
    errors = np.abs(np.round(df_test['pred_position']) - df_test['position'])
    if pts == 10:
        mask = errors == 0
    elif pts == 7:
        mask = (errors > 0) & (errors <=1)
    elif pts == 5:
        mask = (errors > 1) & (errors <=2)
    elif pts == 3:
        mask = (errors > 2) & (errors <=3)
    else:
        mask = errors >3
    print(f'{label}: {mask.sum():>5} predictions ({mask.mean()*100:.1f}%)')


test_results['reward_score'] = {
    'total': total_reward,
    'max': max_reward,
    'pct': reward_pct,
    'avg': avg_reward,
    
}

## Reward Score Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))

#left - reward score distribution
ax1=axes[0]
reward_counts= df_test['reward'].value_counts().sort_index()

bar_colors_r = [
    '#27AE60' if r >= 12 else
    '#3498DB' if r >= 7 else
    '#F5A623' if r >=3 else
    F1_RED
    for r in reward_counts.index
]

ax1.bar(
    reward_counts.index,
    reward_counts.values,
    color=bar_colors_r,
    edgecolor='white',
    linewidth=0.6,
    width=0.7
)

for i, (score, cnt) in enumerate(reward_counts.items()):
    ax1.text(score, cnt + 5, str(cnt),
             ha='center', va ='bottom', fontsize=8, fontweight='bold')
ax1.axvline(avg_reward, color='black', linestyle='--',
            linewidth=1.5, label=f'Average: {avg_reward:.1f}')
ax1.set_xlabel('Reward Points per Driver per Race', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Distribution Reward Scores',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)



#right - reward by season

ax2 = axes[1]
reward_by_season = (
    df_test.groupby('season')['reward']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
)
reward_by_season['efficiency'] = (
    reward_by_season['sum'] /
    (reward_by_season['count'] * max_possible_per_race) *100
)

bars2 = ax2.bar(
    reward_by_season['season'].astype(str),
    reward_by_season['mean'],
    color=['#3468DB', '#E67E22'][:len(reward_by_season)],
    edgecolor='white',
    linewidth=0.6,
    width=0.6
)

for bar, mean_val, eff in zip(
    bars2,
    reward_by_season['mean'],
    reward_by_season['efficiency']
):
    ax2.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() + 0.1,
        f'{mean_val:.2f}\n({eff:.0f}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax2.axvline(avg_reward, color='black', linestyle='--',
            linewidth=1.5, label=f'Overall avg: {avg_reward:.2f}')
ax2.set_ylabel('Avg Reward per Race Entry', fontsize=12)
ax2.set_xlabel('Season', fontsize=12)
ax2.set_title('Average Reward Score by Season',
              fontsize=13, fontweight='bold')
ax2.set_ylim(0, max_possible_per_race + 1)
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle(
    f'Prediction Reward Score - overall Efficiency: {reward_pct:.1f}%',
    fontsize=14, fontweight='bold', y=1.02
)

fig.tight_layout()
fig.savefig('../output/06_reward_score.png', dpi =150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Reward Score by Driver

In [ ]:
reward_by_driver = (
    df_test.groupby('driver_name')['reward']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_reward', 'count': 'races'})
)

reward_by_driver = (
    reward_by_driver[reward_by_driver['races'] >= 5]
    .sort_values('avg_reward', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(13,8))

colors_rwd = [
    '#27AE60' if r >= avg_reward *1.1 else
    '#3468DB' if r >= avg_reward else
    '#F5A623' if r >= avg_reward * 0.85 else
    F1_RED
    for r in reward_by_driver['avg_reward']
]

ax.barh(
    reward_by_driver['driver_name'],
    reward_by_driver['avg_reward'],
    color=colors_rwd,
    edgecolor='white',
    linewidth=0.4,
    height=0.75
)

for i, (rwd, cnt) in enumerate(
    zip(reward_by_driver['avg_reward'], reward_by_driver['races'])

):
    ax.text(
        rwd + 0.05, i,
        f'{rwd:.2f} (n={cnt})',
        va='center', ha='left', fontsize=9
    )

ax.axvline(avg_reward, color='black', linestyle='--',
           linewidth=1.5, label=f'Overall avg: {avg_reward:.2f}')
ax.set_xlabel('Average Reward Score (max 15)', fontsize=12)
ax.set_title('Average Prediction Reward Score by Driver\n'
             '(Higher = more predictable driver)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, max_possible_per_race +1)
ax.legend(fontsize=9)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_reward_by_driver.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# Section 5 - Final Model Report

**Goal:** Consolidate all evaluation results into a single report
comparing validation vs test performance, identifying failure modes,
and making a final recommendation for Notebook 07.

---

### The Three Questions We Answer Here

1. **Did the model generalise?**
   Compare validation metrics (2024) vs test metrics (2025–2026).
   If test metrics are significantly worse → the model overfit to 2024.

2. **Where does the model fail?**
   Identify systematic failure patterns by driver, circuit, and season.
   These inform what additional features could improve future versions.

3. **Is the model ready for Driver Cards?**
   Notebook 07 uses the position model and points model to generate
   driver profiles. We confirm minimum quality thresholds here.


## Validation vs Test Comparison

In [ ]:
# position
val_mae_pos = metadata['position_model']['val_mae']
val_rmse_pos = metadata['position_model']['val_rmse']
val_r2_pos = metadata['position_model']['val_r2']
tst_mae_pos = test_results['position']['MAE']
tst_rmse_pos = test_results['position']['RMSE']
tst_r2_pos = test_results['position']['R2']

delta_mae_pos = tst_mae_pos - val_mae_pos
flag_pos = '✅' if abs(delta_mae_pos) <= 0.5 else '⚠️ GENERALISATION GAP'

print(f'TARGET: position ({metadata['position_model']['name']})')
print(f'{'Metric':<10} {'Validation':>12} {'Test':>12} {'Delta':>10}')
print(f'  {"-"*50}')
print(f'{"MAE":<10} {val_mae_pos:>12.4f} {tst_mae_pos:>12.4f}'
      f'{delta_mae_pos:>+10.4f} {flag_pos}')
print(f'{'RMSE':<10} {val_rmse_pos:>12.4f} {tst_rmse_pos:>12.4f}'
      f'{tst_rmse_pos - val_rmse_pos:>+10.4f}')
print(f'{'R2':<10} {val_r2_pos:>12.4f} {tst_r2_pos:>12.4f}'
      f'{tst_r2_pos - val_r2_pos:>+10.4f}\n')


#points finish
val_f1_clf = metadata['points_finish_model']['val_f1']
val_acc_clf = metadata['points_finish_model']['val_acc']
tst_f1_clf = test_results['points_finish']['F1']
tst_acc_clf = test_results['points_finish']['Accuracy']

delta_f1 = tst_f1_clf - val_f1_clf
flag_clf = '✅' if abs(delta_f1) <= 0.05 else '⚠️ GENERALISATION GAP'

print(f'TARGET: points_finish ({metadata["points_finish_model"]["name"]})')
print(f'{"Metric":<10} {"Validation":>12} {"Test":>12} {"Delta":>10}')
print(f'  {"-"*50}')
print(f'{"F1":<10} {val_f1_clf:>12.4f} {tst_f1_clf:>12.4f}'
      f'{delta_f1:>+10.4f} {flag_clf}')
print(f'{"Accuracy":<10}  {val_acc_clf:>12.4f} {tst_acc_clf:>12.4f}'
      f'{tst_acc_clf - val_acc_clf:>10.4f}')

print()

#Points
val_mae_pts = metadata['points_model']['val_mae']
val_rmse_pts = metadata['points_model']['val_rmse']
val_r2_pts = metadata['points_model']['val_r2']
tst_mae_pts= test_results['points']['MAE']
tst_rmse_pts = test_results['points']['RMSE']
tst_r2_pts = test_results['points']['R2']

delta_mae_pts = tst_mae_pts - val_mae_pts
flag_pts = '✅' if abs(delta_mae_pts) <= 0.5 else '⚠️  GENERALISATION GAP'

print(f'TARGET: points ({metadata["points_model"]["name"]})')
print(f'{"Metric":<10} {"Validation":>12} {"Test":>12} {"Delta":>10}')
print(f'   {"-"*50}')
print(f'{"MAE":<10} {val_mae_pts:>12.4f} {tst_mae_pts:>12.4f}'
      f'{delta_mae_pts:>+10.4f} {flag_pts}')
print(f'{"RMSE":<10} {val_rmse_pts} {tst_rmse_pts:>12.4f}'
      f'{tst_rmse_pts - val_rmse_pts:>10.4f}')
print(f'{"R2":<10} {val_r2_pts:>12.4f} {tst_r2_pts:>12.4f}'
      f'{tst_r2_pts - val_r2_pts}')


## Visual Val vs Test Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

#chart 1 - Position MAE
ax1 = axes[0]
bars1 =ax1.bar(
    ['Validation\n(2024)', 'Test\n(2025-26)'],
    [val_mae_pos, tst_mae_pos],
    color=['#3498DB', F1_RED if delta_mae_pos > 0.5 else '#27AE60'],
    edgecolor='white', linewidth=0.6, width=0.5
)

for bar, val in zip(bars1, [val_mae_pos, tst_mae_pos]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax1.set_ylabel('MAE (positions)', fontsize=11)
ax1.set_title(f'Position Model\nMAE (lower = better)',
              fontsize=12, fontweight='bold')
ax1.set_ylim(0, max(val_mae_pos, tst_mae_pos) * 1.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#chart 2 - points finish F1
ax2 = axes[1]
bars2 = ax2.bar(
    ['Validation\n(2024)', 'Test\n(2025=26)'],
    [val_f1_clf, tst_f1_clf],
    color=['#3498DB', '#27AE60' if delta_f1 >= -0.05 else F1_RED],
    edgecolor='white', linewidth=0.6, width=0.5
)

for bar, val in zip(bars2, [val_f1_clf, tst_f1_clf]):
    ax2.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() + 0.005,
        f'{val:.4f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
    
ax2.set_ylabel('F1 Score', fontsize=11)
ax2.set_title(f'Points Finish Model\nF1 Score (higher = better)',
              fontsize=12, fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

#chart 3 - Points MAE
ax3 = axes[2]
bars3 =ax3.bar(
    ['Validation\n(2024)', 'Test\n(2025-26)'],
    [val_mae_pts, tst_mae_pts],
    color=['#3498DB', F1_RED if delta_mae_pts > 0.5 else '#27AE60'],
    edgecolor='white', linewidth=0.6, width=0.5
)
for bar, val in zip(bars3, [val_mae_pts, tst_mae_pts]):
    ax3.text(
        bar.get_x() + bar.get_width() /2,
        bar.get_height() + 0.02,
        f'{val:.4f}',
        ha='center', va = 'bottom', fontsize=11, fontweight='bold'
    )

ax3.set_ylabel('MAE (points)', fontsize=11)
ax3.set_title(f'Points Model\nMAE (lower = better)',
              fontsize=12, fontweight='bold')
ax3.set_ylim(0, max(val_mae_pts, tst_mae_pts) * 1.3)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

fig.suptitle('Validation vs Test Performance - All Three Models',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('../output/06_val_vs_test_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Failure Mode Analysis

In [ ]:
# top 20 worst position predictions
worst_predictions = (
    df_test[['season', 'round', 'circuit_id', 'driver_name',
             'position', 'pred_position', 'pos_error', 'dnf',
             'constructor_avg_finish_last_3_races']]
    .assign(abs_error=df_test['pos_error'].abs())
    .sort_values('abs_error', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print('TOP 20 WORST POSITION PREDICTIONS:')
print('=' * 90)
print(f'{"#":<4} {"Season":<8} {"Round":<7} {"Circuit":<20} '
      f'{"Driver":<20} {"Actual":>7} {"Pred":>7} {"Error":>7} {"dnf":>5}')
print('-' * 90)

for i, row in worst_predictions.iterrows():
    # Используем row['column'] вместо row.column
    # чтобы избежать конфликта с методами pandas
    print(f'{i+1:<4} {int(row["season"]):<8} {int(row["round"]):<7} '
          f'{row["circuit_id"]:<20} {row["driver_name"]:<20} '
          f'{int(row["position"]):>7} {row["pred_position"]:>7.1f} '
          f'{row["pos_error"]:>+7.1f} {int(row["dnf"]):>5}')

print()
print('Pattern analysis:')
dnf_in_worst = worst_predictions['dnf'].sum()
print(f'  DNFs among worst 20 predictions: {int(dnf_in_worst)} '
      f'({dnf_in_worst/20*100:.0f}%)')
print(f'  Most common circuit in worst 20: '
      f'{worst_predictions["circuit_id"].mode()[0]}')
print()
print('Insight: DNF events are inherently unpredictable from historical data.')
print('A driver retiring from P3 looks like a P18 error to the model.')

## DNF Impact on Metrics

In [ ]:
dnf_mask = df_test['dnf'] == 1
no_dnf_mask = df_test['dnf'] == 0

mae_with_dnf = mean_absolute_error(
    df_test['position'], df_test['pred_position']
)

mae_without_dnf = mean_absolute_error(
    df_test.loc[no_dnf_mask, 'position'],
    df_test.loc[no_dnf_mask, 'pred_position']
) if dnf_mask.sum() > 0 else float('nan')

mae_dnf_only = mean_absolute_error(
    df_test.loc[dnf_mask, 'position'],
    df_test.loc[dnf_mask, 'pred_position']
) if dnf_mask.sum() > 0 else float('nan')

print('\nDNF Impact on Position Prediction Accuracy')
print('='*55)

print(f'\nAll races (including DNFs): MAE = {mae_with_dnf:.4f}')
print(f'Clean races only (no DNF): MAE = {mae_without_dnf:.4f}')
if not np.isnan(mae_dnf_only):
    print(f'DNF races only: MAE = {mae_dnf_only:.4f}')

improvement = (mae_with_dnf - mae_without_dnf) / mae_with_dnf *100
print(f'\nRemoving DNFs improves MAE by: {improvement:.1f}%')

print(f'\nTotal DNFs in test set: {dnf_mask.sum()}'
      f'({dnf_mask.mean()*100:.1f}% of race entries)')

print('\nConclusion:')
if improvement > 10:
    print(f'DNF events significantly inflate the MAE')
    print(f'The model\'s true predictive ability on clean races is')
    print(f'considerably better than the headline MAE suggests')
else:
    print(f'DNF events have moderate impact on headline MAE')
    print(f'The model handles retirement scenatios reasonably well')
    

## Season-Level Race Predictons

In [ ]:
# Accumulate actual and predicted points per driver
standing_actual = (
    df_test.groupby('driver_name')['points']
    .sum()
    .reset_index()
    .rename(columns={'points': 'actual_total'})
    .sort_values('actual_total', ascending=False)
    .reset_index(drop=True)
)

standing_pred = (
    df_test.groupby('driver_name')['pred_points']
    .sum()
    .reset_index()
    .rename(columns={'pred_points': 'predicted_total'})
)

standings = standing_actual.merge(standing_pred, on='driver_name')
standings['rank_actual'] = standings['actual_total'].rank(
    ascending=False, method='min'
).astype(int)
standings['rank_predicted'] = standings['predicted_total'].rank(
    ascending=False, method='min'
).astype(int)
standings['rank_delta'] = standings['rank_actual'] - standings['rank_predicted']


#only show top15
standings_top = standings.head(15)

fig, ax = plt.subplots(figsize=(14,8))

x = np.arange(len(standings_top))
width = 0.38

bars_actual = ax.bar(
    x - width/2,
    standings_top['actual_total'],
    width, label='Actual Points',
    color='#3498DB', edgecolor='white', linewidth=0.5
)

bars_pred = ax.bar(
    x+ width/2,
    standings_top['predicted_total'],
    width, label='Predicted Points',
    color=F1_RED, edgecolor='white', linewidth=0.5, alpha=0.85
)


ax.set_xticks(x)
ax.set_xticklabels(
    standings_top['driver_name'],
    rotation=35, ha='right', fontsize=9
)

ax.set_ylabel('Total Championship Points', fontsize=12)
ax.set_title(
    'Actual vs Predicted Championship Points - Test Set(2026-2026)\n'
    '(How well does the model replicate the season standings?) ',
    fontsize=13, fontweight='bold'
)

ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
fig.savefig('../output/06_championship_standings.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)


In [ ]:
#Ranking accuracy
correct_top3 = len(set(standings[standings.rank_actual <= 3]['driver_name']) &
                   set(standings[standings.rank_predicted <= 3]['driver_name']))
correct_top10 = len(set(standings[standings.rank_actual <= 10]['driver_name']) &
                    set(standings[standings.rank_predicted <= 10]['driver_name']))

print(f'Championship Ranking Accuracy:')
print(f'Correct top-3 drivers: {correct_top3}/3'
      f'({correct_top3/3*100:.0f}%)')
print(f'Correct top-10 drivers: {correct_top10}/10'
      f'({correct_top10/10*100:.0f}%)')

## Save Full Results

In [ ]:
#full test predictions CSV
df_test.to_csv('../output/06_test_predictions.csv', index=False)
print('Test predictions saved ../output/06_test_predictions.csv')

#full metrics summary
eval_summary = {
    'position_model': {
        'name': metadata['points_model']['name'],
        'val_mae': val_mae_pos,
        'test_mae': tst_mae_pos,
        'val_rmse': val_rmse_pos,
        'test_rmse': tst_rmse_pos,
        'val_r2': val_r2_pos,
        'test_r2': tst_r2_pos,
    },
    'points_finish_model': {
        'name': metadata['points_finish_model']['name'],
        'val_f1': val_f1_clf,
        'test_f1': tst_f1_clf,
        'val_acc': val_acc_clf,
        'test_acc': tst_acc_clf,
    },
    'points_model': {
        'name':      metadata['points_model']['name'],
        'val_mae':   val_mae_pts,
        'test_mae':  tst_mae_pts,
        'val_rmse':  val_rmse_pts,
        'test_rmse': tst_rmse_pts,
        'val_r2':    val_r2_pts,
        'test_r2':   tst_r2_pts,

    },
    'reward_score': test_results['reward_score'],
    'dnf_analysis': {
        'mae_all':         mae_with_dnf,
        'mae_no_dnf':      mae_without_dnf,
        'mae_dnf_only':    mae_dnf_only,
        'dnf_improvement': improvement
    },
    'championship': {
        'correct_top3':  correct_top3,
        'correct_top10': correct_top10
    }
}

import pickle
with open('../output/models/eval_summary.pkl', 'wb') as f:
    pickle.dump(eval_summary, f)
print('Evaluation summary saved: ../output/models/eval_summary.pkl')

#Human-readable csv summary
rows=[]

for model_key, m in eval_summary.items():
    if isinstance(m, dict) and 'name' in m:
        row = {'model_target': model_key}
        row.update(m)
        rows.append(row)

pd.DataFrame(rows).to_csv('../output/06_eval_summary.csv', index=False)
print('CSV summary saved: ../output/06_eval_summary.csv')

## List all output files

In [ ]:
import glob

output_files = sorted(
    glob.glob('../output/06_*.png') +
    glob.glob('../output/06_*.csv') + 
    glob.glob('../output/models/eval_summary.pkl')
)

print('Output files generated by Notebook 06:\n')

for f in output_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'📊 {f.split("/")[-1]:<45} ({size_kb:.1f} KB)')

print(f'\nTotal: {len(output_files)} files')


# Summary

## What Was Accomplished

This notebook performed final evaluation of all three models on the
completely held-out test set (seasons 2025–2026), computed the custom
Prediction Reward Score, and diagnosed failure modes.

---

## Results Overview

### Position Model
- Predicts finishing position (1–20) for each driver
- Evaluated by MAE, RMSE, R²
- Broken down by finishing zone, driver, and circuit
- DNF impact quantified separately

### Points Finish Model
- Predicts whether driver scores championship points (binary)
- Evaluated by F1, Accuracy, Precision, Recall
- Confusion matrix and ROC curve generated
- AUC quantifies ranking ability across all thresholds

### Points Model
- Predicts exact championship points scored
- Evaluated by MAE in points
- Bias analysis reveals systematic over/underestimation by driver

### Custom Reward Score
- Domain-specific metric rewarding near-miss predictions
- Computed per driver per race across test set
- Broken down by season and driver

---

## Key Findings

**Generalisation:** Compare validation vs test deltas in the output.
Small deltas confirm the model generalised well. Large deltas suggest
overfitting to the 2024 season.

**DNF Impact:** Retirement events are unpredictable from historical features.
The MAE on clean races (no DNF) is the true measure of model capability.

**Championship Accuracy:** The model correctly identifies the top
championship contenders even if exact points totals vary.


## Saved Output Files

| File | Description |
|------|-------------|
| `06_position_actual_vs_predicted.png` | Scatter + error distribution |
| `06_position_mae_by_zone.png` | MAE by finishing zone |
| `06_position_mae_by_driver.png` | MAE by driver |
| `06_position_mae_by_circuit.png` | MAE by circuit |
| `06_confusion_matrix.png` | Points finish confusion matrix |
| `06_roc_curve.png` | ROC curve + AUC |
| `06_points_actual_vs_predicted.png` | Points scatter + avg by bracket |
| `06_points_error_by_driver.png` | Points MAE + bias by driver |
| `06_reward_score.png` | Reward score distribution |
| `06_reward_by_driver.png` | Reward score by driver |
| `06_val_vs_test_comparison.png` | Val vs test metric comparison |
| `06_championship_standings.png` | Predicted vs actual standings |
| `06_test_predictions.csv` | Full test set with all predictions |
| `06_eval_summary.csv` | Model metrics summary table |
| `eval_summary.pkl` | Full metrics dict for Notebook 07 |

---

## Next Step — Notebook 07: Driver Cards

With validated models and full test predictions available, we now:
1. Generate a complete Driver Card for each driver
2. Identify best and worst circuits per driver
3. Plot performance trends across seasons
4. Compute strength labels (strong qualifier, circuit specialist, etc.)
5. Output a final visual Driver Card per driver
